# Задание 1.

Изучить представленный набор данных на основе описания его столбцов и выбрать 8 столбцов для дальнейшего изучения — среди них должны быть как числовые, так и категориальные. Провести для них расчёт базовых метрик (статистические величины: мода, медиана и пр.), кратко описать результаты.

In [1]:
import pandas as pd


def load_horse_data(path):
    column_names = [
        'surgery', 'age', 'hospital_number', 'rectal_temperature', 'pulse',
        'respiratory_rate', 'temp_of_extremities', 'peripheral_pulse',
        'mucous_membranes', 'capillary_refill_time', 'pain', 'peristalsis',
        'abdominal_distension', 'nasogastric_tube', 'nasogastric_reflux',
        'nasogastric_reflux_ph', 'rectal_exam_feces', 'abdomen',
        'packed_cell_volume', 'total_protein', 'abdom_appearance',
        'abdom_total_protein', 'outcome', 'surgical_lesion',
        'lesion_1', 'lesion_2', 'lesion_3', 'cp_data'
    ]
    return pd.read_csv(path, header=None, names=column_names, na_values='?')


def select_columns(df):
    columns = [
        'surgery', 'age', 'rectal_temperature', 'pulse',
        'respiratory_rate', 'temp_of_extremities', 'pain', 'outcome'
    ]
    return df[columns].copy()


def basic_metrics(df, numeric_cols, categorical_cols):
    print('Числовые столбцы:')
    print(df[numeric_cols].describe())
    print()
    print('Медиана:')
    print(df[numeric_cols].median())
    print()
    print('Мода:')
    for col in df.columns:
        mode_value = df[col].mode(dropna=True)
        if not mode_value.empty:
            print(f'{col}: {mode_value.iloc[0]}')
    print()
    print('Категориальные столбцы (частоты):')
    for col in categorical_cols:
        print(f'\n{col}:')
        print(df[col].value_counts(dropna=False))


df_raw = load_horse_data('horse_data.csv')
df = select_columns(df_raw)

numeric_cols = ['rectal_temperature', 'pulse', 'respiratory_rate']
categorical_cols = ['surgery', 'age', 'temp_of_extremities', 'pain', 'outcome']

print(df.head())
print()
basic_metrics(df, numeric_cols, categorical_cols)

   surgery  age  rectal_temperature  pulse  respiratory_rate  \
0      2.0    1                38.5   66.0              28.0   
1      1.0    1                39.2   88.0              20.0   
2      2.0    1                38.3   40.0              24.0   
3      1.0    9                39.1  164.0              84.0   
4      2.0    1                37.3  104.0              35.0   

   temp_of_extremities  pain  outcome  
0                  3.0   5.0      2.0  
1                  NaN   3.0      3.0  
2                  1.0   3.0      1.0  
3                  4.0   2.0      2.0  
4                  NaN   NaN      2.0  

Числовые столбцы:
       rectal_temperature       pulse  respiratory_rate
count          240.000000  276.000000        242.000000
mean            38.167917   71.913043         30.417355
std              0.732289   28.630557         17.642231
min             35.400000   30.000000          8.000000
25%             37.800000   48.000000         18.500000
50%             38.2

Выбраны столбцы:

числовые — rectal_temperature, pulse, respiratory_rate;

категориальные — surgery, age, temp_of_extremities, pain, outcome.

Кратко по результатам:

rectal_temperature — средняя около 38.2°C, близка к норме 37.8°C; разброс небольшой.

pulse — медиана 64 при норме 30–40, у многих лошадей пульс повышен (боль/шок).

respiratory_rate — медиана 24.5 при норме 8–10, показатель сильно колеблется.

surgery — чаще значение 1 (была операция).

age — в данных встречаются 1 и 9; по описанию молодые должны кодироваться как 2, значит 9 — ошибка кодирования.

temp_of_extremities — чаще 3 (cool), что может говорить о нарушении кровообращения.

pain — чаще 3 (умеренная перемежающаяся боль).

outcome — чаще 1 (лошадь выжила).

# Задание 2.

В выбранных числовых столбцах найти выбросы, выдвинуть гипотезы об их причинах и проинтерпретировать результаты. Принять и обосновать решение о дальнейшей работе с ними.

In [2]:
def find_outliers_iqr(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = series[(series < lower) | (series > upper)]
    return outliers, lower, upper


def report_outliers(df, numeric_cols):
    for col in numeric_cols:
        outliers, lower, upper = find_outliers_iqr(df[col].dropna())
        print(f'{col}:')
        print(f'границы IQR: [{lower:.2f}; {upper:.2f}]')
        print(f'количество выбросов: {len(outliers)}')
        print(f'значения: {sorted(outliers.tolist())}')
        print()


report_outliers(df, numeric_cols)

rectal_temperature:
границы IQR: [36.75; 39.55]
количество выбросов: 14
значения: [35.4, 36.0, 36.1, 36.4, 36.5, 36.5, 36.6, 39.6, 39.7, 39.9, 40.0, 40.3, 40.3, 40.8]

pulse:
границы IQR: [-12.00; 148.00]
количество выбросов: 5
значения: [150.0, 150.0, 160.0, 164.0, 184.0]

respiratory_rate:
границы IQR: [-7.75; 62.25]
количество выбросов: 17
значения: [66.0, 68.0, 68.0, 68.0, 70.0, 70.0, 72.0, 80.0, 80.0, 80.0, 84.0, 84.0, 88.0, 90.0, 90.0, 96.0, 96.0]



Интерпретация выбросов:

rectal_temperature — низкие значения (около 35–36°C) могут соответствовать шоку, высокие (около 40°C) — инфекции. Это клинически возможные состояния, а не ошибки ввода.

pulse — выбросы только сверху (150–184). Высокий пульс типичен при сильной боли и циркуляторном шоке; также у молодых лошадей пульс в норме выше.

respiratory_rate — много верхних выбросов. По описанию датасета показатель малополезен из-за сильных колебаний, поэтому выбросы здесь ожидаемы.

Решение: выбросы не удаляю. Это данные о больных лошадях, экстремальные значения отражают тяжёлое состояние и важны для анализа. Удаление исказило бы картину.

# Задание 3.

Рассчитать количество пропусков для всех выбранных столбцов. Принять и обосновать решение о методе работы с пропусками по каждому столбцу, сформировать датафрейм, в котором пропусков не будет.

In [3]:
def missing_report(df):
    report = pd.DataFrame({
        'пропуски': df.isna().sum(),
        'доля': df.isna().mean().round(3)
    })
    print(report)


def fill_missing(df):
    result = df.copy()

    # age: в описании young = 2, в данных ошибочно стоит 9
    result['age'] = result['age'].replace(9, 2)

    # числовые — медиана (устойчива к выбросам)
    for col in ['rectal_temperature', 'pulse', 'respiratory_rate']:
        result[col] = result[col].fillna(result[col].median())

    # категориальные — мода
    for col in ['surgery', 'temp_of_extremities', 'pain', 'outcome']:
        result[col] = result[col].fillna(result[col].mode().iloc[0])

    return result


print('Пропуски до обработки:')
missing_report(df)

df_clean = fill_missing(df)

print('\nПропуски после обработки:')
missing_report(df_clean)
print()
print(df_clean.head())
print()
print('Размер датафрейма:', df_clean.shape)

Пропуски до обработки:
                     пропуски   доля
surgery                     1  0.003
age                         0  0.000
rectal_temperature         60  0.200
pulse                      24  0.080
respiratory_rate           58  0.193
temp_of_extremities        56  0.187
pain                       55  0.183
outcome                     1  0.003

Пропуски после обработки:
                     пропуски  доля
surgery                     0   0.0
age                         0   0.0
rectal_temperature          0   0.0
pulse                       0   0.0
respiratory_rate            0   0.0
temp_of_extremities         0   0.0
pain                        0   0.0
outcome                     0   0.0

   surgery  age  rectal_temperature  pulse  respiratory_rate  \
0      2.0    1                38.5   66.0              28.0   
1      1.0    1                39.2   88.0              20.0   
2      2.0    1                38.3   40.0              24.0   
3      1.0    2                39.1 

Решение по пропускам:

surgery, outcome — по 1 пропуску, заполняю модой (самое частое значение).

age — пропусков нет, но значение 9 заменяю на 2 (ошибка кодирования молодых лошадей).

rectal_temperature, pulse, respiratory_rate — пропусков много, заполняю медианой, чтобы меньше влияли выбросы.

temp_of_extremities, pain — категориальные признаки, заполняю модой.

Строки не удаляю, пропусков слишком много, удаление сильно уменьшит выборку.